In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from catboost import CatBoostClassifier, Pool

# Baseline

## Train

In [2]:
train = pd.read_csv('../data/900-train.csv')

In [10]:
X = train.columns[3:-1]  # Все кроме 'id', 'CustomerId', 'Surname', 'Exited'
cat_features = ['Geography', 'Gender']
y = train.columns[-1:]  # Exited

In [11]:
train_data, valid_data = train_test_split(train, train_size=0.8, random_state=42, stratify=train[y])

In [12]:
train_pool = Pool(data=train_data[X],
                  label=train_data[y],
                  cat_features=cat_features
                  )

valid_pool = Pool(data=valid_data[X],
                  label=valid_data[y],
                  cat_features=cat_features
                  )

In [13]:
params = {
    'iterations': 1000,
    'learning_rate': 0.1,
    'eval_metric': 'AUC',
    'verbose': 100,
    'random_seed': 42,
}

model = CatBoostClassifier(**params)
model.fit(train_pool, eval_set=valid_pool)

0:	test: 0.8838775	best: 0.8838775 (0)	total: 66.5ms	remaining: 1m 6s
100:	test: 0.9284950	best: 0.9285146 (97)	total: 1.88s	remaining: 16.8s
200:	test: 0.9273922	best: 0.9285146 (97)	total: 4.52s	remaining: 18s
300:	test: 0.9259954	best: 0.9285146 (97)	total: 7.09s	remaining: 16.5s
400:	test: 0.9247705	best: 0.9285146 (97)	total: 8.93s	remaining: 13.3s
500:	test: 0.9240057	best: 0.9285146 (97)	total: 11.8s	remaining: 11.7s
600:	test: 0.9229076	best: 0.9285146 (97)	total: 14.2s	remaining: 9.44s
700:	test: 0.9216983	best: 0.9285146 (97)	total: 17.6s	remaining: 7.49s
800:	test: 0.9207587	best: 0.9285146 (97)	total: 20.3s	remaining: 5.04s
900:	test: 0.9189653	best: 0.9285146 (97)	total: 21.6s	remaining: 2.37s
999:	test: 0.9189532	best: 0.9285146 (97)	total: 22.7s	remaining: 0us

bestTest = 0.9285145796
bestIteration = 97

Shrink model to first 98 iterations.


In [14]:
y_pred_proba = model.predict_proba(valid_data[X])[:, 1]
roc_auc = roc_auc_score(valid_data[y], y_pred_proba)
print(f"Значение ROC-AUC на валидационном наборе: {roc_auc}")

Значение ROC-AUC на валидационном наборе: 0.9285145795562461


## Test

In [15]:
test = pd.read_csv('../data/900-test.csv')

In [16]:
test_pred_proba = model.predict_proba(test[X])[:, 1]

In [21]:
submission = pd.DataFrame({
    'id': test['id'],            # или другое название для ID, как указано в соревновании
    'Exited': test_pred_proba
})

In [22]:
submission.to_csv('../data/900-submission-v1.csv', index=False)